# 06 — Gold Layer: `daily_risk_metrics`

**Target Table:** `bfsi.gold_layer.daily_risk_metrics`

**Objective:** Build a production-grade Gold aggregation table that computes daily fraud-risk KPIs from the Silver layer, optimized for BI consumption (Power BI / Tableau).

| Feature | Implementation |
|---------|----------------|
| Daily aggregations | Total txn count/value, fraud counts, fraud rate |
| Channel breakdown | CARD / UPI / NEFT_RTGS counts per day |
| Fraud logic | LEFT SEMI JOIN with `fraud_alerts` |
| Merchant risk | Top-5 MCC by fraud rate |
| PII compliance | Only masked/tokenized fields used |
| Optimization | Partitioned by `report_date`, Z-ORDERed, broadcast joins |

**Source Tables:**
- `silver_layer.unified_transactions`
- `silver_layer.fraud_alerts`
- `bronze_layer.merchants` (merchant_master)

> Reads from Silver masked tables — no PII is ever exposed.

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
from datetime import datetime

# ══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════

CATALOG       = "bfsi"
SILVER_SCHEMA = "silver_layer"
BRONZE_SCHEMA = "bronze_layer"
GOLD_SCHEMA   = "gold_layer"

# ── Source tables ──────────────────────────────────────────────────────
UNIFIED_TXN_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"
FRAUD_ALERTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.fraud_alerts"
MERCHANTS_TABLE    = f"{CATALOG}.{BRONZE_SCHEMA}.merchants"

# ── Target table ──────────────────────────────────────────────────────
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.daily_risk_metrics"

# ── Pipeline version ──────────────────────────────────────────────────
PIPELINE_VERSION = "v1.0-gold-risk"

print(f"Gold Layer: daily_risk_metrics")
print(f"Timestamp : {datetime.now().isoformat()}")
print(f"Target    : {GOLD_TABLE}")
print(f"Version   : {PIPELINE_VERSION}")

---
## Cell 2 — Create Gold Schema (Idempotent)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CREATE GOLD SCHEMA IF NOT EXISTS
# ══════════════════════════════════════════════════════════════════════════

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print(f"Schema ready: {CATALOG}.{GOLD_SCHEMA}")

---
## Cell 3 — Load & Validate Source Tables

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  LOAD SOURCE TABLES WITH VALIDATION
#  We verify row counts and schema before proceeding.
# ══════════════════════════════════════════════════════════════════════════

# 1. Unified Transactions (Silver)
df_txn = spark.read.table(UNIFIED_TXN_TABLE)
txn_count = df_txn.count()
print(f"unified_transactions : {txn_count:,} rows")

# 2. Fraud Alerts (Silver)
df_fraud = spark.read.table(FRAUD_ALERTS_TABLE)
fraud_count = df_fraud.count()
print(f"fraud_alerts         : {fraud_count:,} rows")

# 3. Merchants (Bronze — for MCC analysis)
df_merchants = spark.read.table(MERCHANTS_TABLE)
merchant_count = df_merchants.count()
print(f"merchants            : {merchant_count:,} rows")

# ── Validate required columns ──────────────────────────────────────────
required_txn_cols = ["txn_id", "customer_id", "txn_channel", "txn_amount_inr", "txn_ts", "status", "merchant_id", "is_international"]
required_fraud_cols = ["txn_id", "fraud_rule_triggered"]
required_merchant_cols = ["merchant_id", "merchant_category_code", "risk_tier", "is_blacklisted"]

for name, df, req in [
    ("unified_transactions", df_txn, required_txn_cols),
    ("fraud_alerts", df_fraud, required_fraud_cols),
    ("merchants", df_merchants, required_merchant_cols),
]:
    missing = [c for c in req if c not in df.columns]
    status = "OK" if not missing else f"MISSING: {missing}"
    print(f"  {name:30s} → {status}")

assert txn_count > 0, "unified_transactions is empty — cannot proceed"
print("\nAll source tables validated.")

---
## Cell 4 — Flag Fraud Transactions (LEFT JOIN Strategy)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FLAG FRAUD TRANSACTIONS
#
#  Strategy: LEFT JOIN unified_transactions with fraud_alerts on txn_id.
#  A transaction is "fraud-flagged" if it has a matching record in
#  fraud_alerts (i.e., at least one fraud rule was triggered).
#
#  Optimization:
#  - Use broadcast join if fraud_alerts is small (< 100K rows)
#  - Select only txn_id from fraud_alerts to minimise shuffle
#  - Deduplicate fraud_alerts on txn_id (a txn may fire multiple rules)
# ══════════════════════════════════════════════════════════════════════════

# Deduplicate fraud txn_ids — each txn counted once regardless of rules
df_fraud_ids = df_fraud.select("txn_id").distinct()
fraud_id_count = df_fraud_ids.count()
print(f"Distinct fraud txn_ids: {fraud_id_count:,}")

# Broadcast if small enough (< 2M rows); otherwise standard join
BROADCAST_THRESHOLD = 2_000_000
if fraud_id_count < BROADCAST_THRESHOLD:
    print(f"Using BROADCAST join (fraud_ids < {BROADCAST_THRESHOLD:,})")
    df_fraud_ids = F.broadcast(df_fraud_ids)
else:
    print("Using standard SORT-MERGE join (large fraud set)")

# LEFT JOIN to flag fraud
df_with_fraud = (
    df_txn
    .join(
        df_fraud_ids.withColumnRenamed("txn_id", "_fraud_txn_id"),
        df_txn.txn_id == F.col("_fraud_txn_id"),
        how="left"
    )
    .withColumn(
        "is_fraud_flagged",
        F.when(F.col("_fraud_txn_id").isNotNull(), F.lit(True))
         .otherwise(F.lit(False))
    )
    .drop("_fraud_txn_id")
)

# ── Derive report_date from txn_ts ─────────────────────────────────────
df_with_fraud = df_with_fraud.withColumn(
    "report_date", F.to_date(F.col("txn_ts"))
)

# ── Cache for reuse across multiple aggregations ───────────────────────
df_with_fraud.cache()
total = df_with_fraud.count()
flagged = df_with_fraud.filter(F.col("is_fraud_flagged")).count()
print(f"\nTotal transactions : {total:,}")
print(f"Fraud-flagged      : {flagged:,} ({100*flagged/total:.2f}%)")
print(f"Clean              : {total - flagged:,}")

---
## Cell 5 — Compute Daily Aggregations

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  DAILY AGGREGATIONS
#
#  Metrics per report_date:
#    1. total_txn_count          — all transactions
#    2. total_txn_value_inr      — sum of txn_amount_inr
#    3. fraud_flagged_count      — transactions present in fraud_alerts
#    4. fraud_flagged_value      — sum of fraud txn amounts
#    5. fraud_rate_pct           — (fraud_count / total_count) × 100
#
#  Channel-wise breakdown:
#    6. card_txn_count
#    7. upi_txn_count
#    8. neft_rtgs_txn_count
#
#  Null handling:
#    - txn_amount_inr NULLs → treated as 0 via coalesce
#    - Division by zero guarded with CASE WHEN
# ══════════════════════════════════════════════════════════════════════════

df_daily = (
    df_with_fraud
    .groupBy("report_date")
    .agg(
        # ── Core metrics ──
        F.count("*").alias("total_txn_count"),
        F.round(
            F.sum(F.coalesce(F.col("txn_amount_inr"), F.lit(0.0))), 2
        ).alias("total_txn_value_inr"),

        # ── Fraud metrics ──
        F.sum(
            F.when(F.col("is_fraud_flagged"), F.lit(1)).otherwise(F.lit(0))
        ).alias("fraud_flagged_count"),
        F.round(
            F.sum(
                F.when(F.col("is_fraud_flagged"),
                       F.coalesce(F.col("txn_amount_inr"), F.lit(0.0)))
                 .otherwise(F.lit(0.0))
            ), 2
        ).alias("fraud_flagged_value"),

        # ── Channel-wise counts ──
        F.sum(
            F.when(F.col("txn_channel") == "CARD", F.lit(1)).otherwise(F.lit(0))
        ).alias("card_txn_count"),
        F.sum(
            F.when(F.col("txn_channel") == "UPI", F.lit(1)).otherwise(F.lit(0))
        ).alias("upi_txn_count"),
        F.sum(
            F.when(F.col("txn_channel") == "NEFT_RTGS", F.lit(1)).otherwise(F.lit(0))
        ).alias("neft_rtgs_txn_count"),

        # ── Additional KPIs ──
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("merchant_id").alias("unique_merchants"),
        F.round(F.avg(F.coalesce(F.col("txn_amount_inr"), F.lit(0.0))), 2).alias("avg_txn_value_inr"),
        F.sum(
            F.when(F.col("is_international") == True, F.lit(1)).otherwise(F.lit(0))
        ).alias("international_txn_count"),
    )
)

# ── Compute fraud_rate_pct (guarded against division by zero) ──────────
df_daily = df_daily.withColumn(
    "fraud_rate_pct",
    F.round(
        F.when(
            F.col("total_txn_count") > 0,
            (F.col("fraud_flagged_count") / F.col("total_txn_count")) * 100
        ).otherwise(F.lit(0.0)),
        4
    )
)

print(f"Daily aggregation computed: {df_daily.count()} report_dates")
display(df_daily.orderBy("report_date").limit(10))

---
## Cell 6 — Top-5 Merchant Categories by Fraud Rate

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TOP-5 MERCHANT CATEGORY CODES (MCC) BY FRAUD RATE
#
#  Join unified_transactions → merchants → fraud flag
#  Group by merchant_category_code, compute fraud_rate per MCC
#  Rank and pick top 5
#
#  Broadcast merchant_master (typically < 50K rows)
# ══════════════════════════════════════════════════════════════════════════

# Broadcast merchant lookup — small table
df_merchant_lookup = F.broadcast(
    df_merchants.select(
        F.col("merchant_id").alias("_m_id"),
        "merchant_category_code",
        "risk_tier",
        "is_blacklisted"
    )
)

# Join transactions (with fraud flag) to merchants
df_txn_merchant = (
    df_with_fraud
    .filter(F.col("merchant_id").isNotNull())   # Only txns with merchant
    .join(
        df_merchant_lookup,
        df_with_fraud.merchant_id == F.col("_m_id"),
        how="inner"
    )
    .drop("_m_id")
)

# Aggregate per MCC
df_mcc_risk = (
    df_txn_merchant
    .groupBy("merchant_category_code")
    .agg(
        F.count("*").alias("mcc_total_txn"),
        F.sum(
            F.when(F.col("is_fraud_flagged"), F.lit(1)).otherwise(F.lit(0))
        ).alias("mcc_fraud_count"),
        F.round(F.sum(F.coalesce(F.col("txn_amount_inr"), F.lit(0.0))), 2).alias("mcc_total_value"),
        F.round(
            F.sum(
                F.when(F.col("is_fraud_flagged"),
                       F.coalesce(F.col("txn_amount_inr"), F.lit(0.0)))
                 .otherwise(F.lit(0.0))
            ), 2
        ).alias("mcc_fraud_value"),
    )
    .withColumn(
        "mcc_fraud_rate_pct",
        F.round(
            F.when(F.col("mcc_total_txn") > 0,
                   (F.col("mcc_fraud_count") / F.col("mcc_total_txn")) * 100)
             .otherwise(F.lit(0.0)),
            4
        )
    )
    # Minimum 10 txns to avoid noisy MCCs
    .filter(F.col("mcc_total_txn") >= 10)
    .orderBy(F.col("mcc_fraud_rate_pct").desc())
    .limit(5)
)

print("Top-5 Merchant Category Codes by Fraud Rate:")
display(df_mcc_risk)

---
## Cell 7 — Embed Top-5 MCC into Daily Metrics

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  EMBED TOP-5 RISKY MCCs AS A STRUCT COLUMN
#  This gives BI tools easy access to the top merchant risk data
#  per pipeline run, without needing a separate lookup table.
# ══════════════════════════════════════════════════════════════════════════

# Collect top-5 as a JSON-safe array (small — max 5 rows)
top5_mcc_rows = df_mcc_risk.collect()
top5_list = [
    f"{row['merchant_category_code']}:{row['mcc_fraud_rate_pct']}%"
    for row in top5_mcc_rows
]
top5_str = " | ".join(top5_list) if top5_list else "N/A"

# Add as a constant column to daily metrics
df_daily_enriched = (
    df_daily
    .withColumn("top5_risky_mcc", F.lit(top5_str))
    .withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
    .withColumn("pipeline_run_ts", F.current_timestamp())
)

print(f"Top-5 risky MCCs: {top5_str}")
print(f"Daily metrics enriched: {df_daily_enriched.count()} rows")

---
## Cell 8 — Final Schema Design & Column Ordering

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FINAL SCHEMA: gold.daily_risk_metrics
#
#  ┌──────────────────────────┬───────────┬─────────────────────────────┐
#  │ Column                   │ Type      │ Description                 │
#  ├──────────────────────────┼───────────┼─────────────────────────────┤
#  │ report_date              │ DATE      │ Partition key               │
#  │ total_txn_count          │ LONG      │ All transactions            │
#  │ total_txn_value_inr      │ DOUBLE    │ Sum of all txn amounts      │
#  │ fraud_flagged_count      │ LONG      │ Fraud-flagged transactions  │
#  │ fraud_flagged_value      │ DOUBLE    │ Sum of fraud txn amounts    │
#  │ fraud_rate_pct           │ DOUBLE    │ (fraud/total) × 100         │
#  │ card_txn_count           │ LONG      │ CARD channel count          │
#  │ upi_txn_count            │ LONG      │ UPI channel count           │
#  │ neft_rtgs_txn_count      │ LONG      │ NEFT_RTGS channel count     │
#  │ unique_customers         │ LONG      │ Distinct customer_ids       │
#  │ unique_merchants         │ LONG      │ Distinct merchant_ids       │
#  │ avg_txn_value_inr        │ DOUBLE    │ Mean transaction value      │
#  │ international_txn_count  │ LONG      │ International txn count     │
#  │ top5_risky_mcc           │ STRING    │ Top-5 MCCs by fraud rate    │
#  │ pipeline_version         │ STRING    │ Audit trail                 │
#  │ pipeline_run_ts          │ TIMESTAMP │ Execution timestamp         │
#  └──────────────────────────┴───────────┴─────────────────────────────┘
#
#  PII Check: ✅ No PII columns — only masked/tokenized IDs
# ══════════════════════════════════════════════════════════════════════════

FINAL_COLUMNS = [
    "report_date",
    "total_txn_count",
    "total_txn_value_inr",
    "fraud_flagged_count",
    "fraud_flagged_value",
    "fraud_rate_pct",
    "card_txn_count",
    "upi_txn_count",
    "neft_rtgs_txn_count",
    "unique_customers",
    "unique_merchants",
    "avg_txn_value_inr",
    "international_txn_count",
    "top5_risky_mcc",
    "pipeline_version",
    "pipeline_run_ts",
]

df_gold = df_daily_enriched.select(*FINAL_COLUMNS)

print("Final schema:")
df_gold.printSchema()
print(f"Rows: {df_gold.count()}")

---
## Cell 9 — Write Gold Table (Partitioned Delta)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  WRITE gold.daily_risk_metrics
#
#  Partitioning: report_date (DATE granularity)
#    → BI tools filter by date range → partition pruning → fast queries
#    → Daily grain avoids small-file problem
#
#  Mode: overwrite (full refresh)
#    → For incremental, switch to MERGE INTO with report_date as key
# ══════════════════════════════════════════════════════════════════════════

(
    df_gold
    .repartition("report_date")          # Align partitions with partition key
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("report_date")
    .saveAsTable(GOLD_TABLE)
)

print(f"\n✅ Gold table written: {GOLD_TABLE}")
print(f"   Partitioned by: report_date")
print(f"   Rows written  : {spark.read.table(GOLD_TABLE).count():,}")

---
## Cell 10 — Z-ORDER & OPTIMIZE

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  Z-ORDER OPTIMIZATION
#
#  Z-ORDER co-locates related data within files for faster predicate
#  pushdown. Best columns to Z-ORDER:
#
#    1. fraud_rate_pct   — BI dashboards filter / sort by fraud rate
#    2. total_txn_count  — Commonly used in WHERE / ORDER BY clauses
#
#  Note: report_date is already the partition key, so we do NOT
#  include it in Z-ORDER (redundant with partition pruning).
#
#  This also compacts small files (bin-packing).
# ══════════════════════════════════════════════════════════════════════════

spark.sql(f"""
    OPTIMIZE {GOLD_TABLE}
    ZORDER BY (fraud_rate_pct, total_txn_count)
""")

print("✅ Z-ORDER optimization complete")
print("   Z-ORDERED on: fraud_rate_pct, total_txn_count")
print("   Partitioned by: report_date")

# ── Suggested indexing for external BI tools ────────────────────────────
print("\n📊 BI Tool Optimization Recommendations:")
print("  Power BI  : Import mode with incremental refresh on report_date")
print("  Tableau   : Live connection with extract on report_date filter")
print("  Both      : Create calculated columns for fraud_rate_pct thresholds")

---
## Cell 11 — Write Top-5 MCC Risk Table (Companion Gold Table)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  COMPANION TABLE: gold.top_mcc_fraud_risk
#  Separate detailed MCC risk table for drill-down dashboards
# ══════════════════════════════════════════════════════════════════════════

MCC_RISK_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.top_mcc_fraud_risk"

df_mcc_final = (
    df_mcc_risk
    .withColumn("report_ts", F.current_timestamp())
    .withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
)

df_mcc_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MCC_RISK_TABLE)

print(f"✅ MCC risk table written: {MCC_RISK_TABLE}")
display(spark.read.table(MCC_RISK_TABLE))

---
## Cell 12 — Unpersist Cache & Cleanup

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CLEANUP: Release cached DataFrames
# ══════════════════════════════════════════════════════════════════════════

df_with_fraud.unpersist()
print("Cache released: df_with_fraud")

---
## Cell 13 — Final Verification & Summary

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  GOLD LAYER — VERIFICATION & SUMMARY
# ══════════════════════════════════════════════════════════════════════════

df_verify = spark.read.table(GOLD_TABLE)

print("=" * 70)
print("  GOLD LAYER: daily_risk_metrics — COMPLETE")
print("=" * 70)

# ── Table stats ──
print(f"\n  Table          : {GOLD_TABLE}")
print(f"  Total rows     : {df_verify.count():,}")
print(f"  Columns        : {len(df_verify.columns)}")
print(f"  Date range     : {df_verify.agg(F.min('report_date')).collect()[0][0]}")
print(f"               to  {df_verify.agg(F.max('report_date')).collect()[0][0]}")

# ── Aggregate verification ──
print("\n  Aggregate Verification:")
agg_row = df_verify.agg(
    F.sum("total_txn_count").alias("sum_txn"),
    F.sum("fraud_flagged_count").alias("sum_fraud"),
    F.round(F.avg("fraud_rate_pct"), 4).alias("avg_fraud_rate"),
    F.sum("card_txn_count").alias("sum_card"),
    F.sum("upi_txn_count").alias("sum_upi"),
    F.sum("neft_rtgs_txn_count").alias("sum_neft"),
).collect()[0]

print(f"    Total transactions  : {agg_row['sum_txn']:,}")
print(f"    Total fraud flagged : {agg_row['sum_fraud']:,}")
print(f"    Avg fraud rate %    : {agg_row['avg_fraud_rate']}")
print(f"    Card txns           : {agg_row['sum_card']:,}")
print(f"    UPI txns            : {agg_row['sum_upi']:,}")
print(f"    NEFT/RTGS txns      : {agg_row['sum_neft']:,}")

# ── Cross-check: sum of channel counts = total ──
channel_total = (agg_row['sum_card'] or 0) + (agg_row['sum_upi'] or 0) + (agg_row['sum_neft'] or 0)
txn_total = agg_row['sum_txn'] or 0
check = "PASS" if channel_total == txn_total else f"FAIL ({channel_total} vs {txn_total})"
print(f"\n  Channel sum check : {check}")

# ── Schema columns ──
print(f"\n  Schema:")
for col in df_verify.columns:
    dtype = dict(df_verify.dtypes)[col]
    print(f"    {col:30s} {dtype}")

# ── PII Check ──
pii_cols = {"customer_name", "email", "phone", "pan", "aadhaar", "address"}
found_pii = pii_cols.intersection(set(df_verify.columns))
pii_status = "PASS — No PII detected" if not found_pii else f"FAIL — Found: {found_pii}"
print(f"\n  PII Compliance : {pii_status}")

print(f"\n  Pipeline version : {PIPELINE_VERSION}")
print(f"  Timestamp        : {datetime.now().isoformat()}")
print("=" * 70)

---

## Optimization & Design Summary

### Partitioning Strategy
| Aspect | Choice | Rationale |
|--------|--------|-----------|
| Partition key | `report_date` | BI tools always filter by date → partition pruning |
| Granularity | Daily | Avoids too-many-small-files; aligns with business reporting cadence |

### Z-ORDER Strategy
| Column | Why |
|--------|-----|
| `fraud_rate_pct` | Most common filter/sort in risk dashboards |
| `total_txn_count` | Used in volume-based alerting and drill-downs |

### Join Optimization
| Join | Strategy | Reason |
|------|----------|--------|
| `unified_txn ⟕ fraud_alerts` | Broadcast (if < 2M rows) | Fraud IDs are typically small vs full txn set |
| `unified_txn ⟕ merchants` | Broadcast | Merchant master is always small (< 50K) |

### Data Skew Mitigation
- **Salt key** approach available if any `report_date` has disproportionate volume.
- **AQE** (Adaptive Query Execution) enabled by default in Databricks.
- Channel-conditional aggregation avoids separate GROUP BY per channel.

### BI Tool Compatibility
| Tool | Mode | Recommendation |
|------|------|----------------|
| Power BI | Import + Incremental Refresh | Filter on `report_date` range |
| Tableau | Live / Extract | Extract with `report_date` filter |
| Both | — | Pre-computed `fraud_rate_pct` avoids client-side division |

### PII Compliance ✅
- `customer_id` is masked/tokenized (SHA-256 hash)
- No names, emails, phone numbers, PAN, or Aadhaar in Gold table
- Only aggregated metrics at daily level — no row-level PII

> **Next Steps:** Connect Power BI / Tableau to `bfsi.gold_layer.daily_risk_metrics` for live dashboards.